kernel : Python (Pixi)

In [16]:
import anndata
import gc
import os
import pandas as pd
from anndata import AnnData
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir

anndata.settings.allow_write_nullable_strings = True

raw_count_matrix_location = find_env_dir("RAW_COUNT_MATRIX")
h5ad_matrix_location = find_env_dir("H5AD_MATRIX")

def load_rds_data(rds_path: str, meta_path: str, series: str) -> AnnData:
    import anndata2ri
    from rpy2.robjects import r
    from rpy2.robjects.conversion import localconverter
    from rpy2.robjects.packages import importr

    importr("base")
    importr("Matrix")
    importr("SingleCellExperiment")

    rcode = f"""
    env_mat <- new.env()
    load(gzcon(gzfile("{rds_path}", "rb")), envir=env_mat)
    mat_name <- ls(env_mat)[1] 
    mat <- env_mat[[mat_name]]
    mat <- as(mat, "dgCMatrix")

    env_meta <- new.env()
    load(gzcon(gzfile("{meta_path}", "rb")), envir=env_meta)
    meta_name <- ls(env_meta)[1]
    meta <- env_meta[[meta_name]]

    meta <- meta[colnames(mat), , drop=FALSE]
    obj <- SingleCellExperiment(
        assays = list(counts = mat),
        colData = meta
    )

    rm(env_mat, env_meta, mat, meta)
    gc()

    obj
    """
    with localconverter(anndata2ri.converter):
        adata = r(rcode)
        r("if (exists('obj')) rm(obj)")
        r("gc()")

    if not isinstance(adata, AnnData):
        raise ValueError(f"Result is not AnnData. Got type: {type(adata)}")

    adata.layers.pop("counts", None)
    if not isinstance(adata.X, csr_matrix):
        adata.X = csr_matrix(adata.X)
    
    # Multidimensional representations (obsm, varm, layers) are intentionally cleared at this stage,
    # since integrated multi-modal / multi-dimensional analyses will be performed after dataset integration, 
    # overwriting any existing representations.
    adata.obsm = {}
    adata.varm = {}
    adata.layers = {}
    gc.collect()

    adata.obs["series"] = series
    return adata

In [ ]:
FILE = "GSE266654_snRNAseq_dgCMatrix_merge.RData.gz"
META = "GSE266654_snRNAseq_metadata_merge.RData.gz"
SERIES_NAME = "Lin"
SAVE_FILE_NAME = SERIES_NAME + ".h5ad"
adata = load_rds_data(os.path.join(raw_count_matrix_location, SERIES_NAME, FILE), meta_path = os.path.join(raw_count_matrix_location, SERIES_NAME, META), series = SERIES_NAME)
adata.obs.head() #type: ignore

Exception ignored from cffi callback <function _processevents at 0x72ae394a61f0>:
Traceback (most recent call last):
  File "/home/sjkim/protocols/.pixi/envs/default/lib/python3.14/site-packages/rpy2/rinterface_lib/callbacks.py", line 308, in _processevents
    @ffi_proxy.callback(ffi_proxy._processevents_def,
KeyboardInterrupt: 


,orig.ident,nCount_RNA,nFeature_RNA,percent.mt,IL01_uniqueID,IL02_species,IL03_source,IL04_sex,IL05_ageDays,IL06_tissue,...,IL08_condition.tissue.status,IL09_illumina,IL10_chemistry,IL11_batch,IL12_LMinDays,IL13_LMaxDays,IL14_dataset,IL15_annotation,IL15_annotation.marker,series
v3.1.1_AAGATAGAGTCTTCGA,v3.1.1,4153.033531,2288,0.024703,v3.1.1,cjacchus,CJH01,F,2051,03fWM,...,fWM_He.Control,Hi4000,10xv3,1st,0.0,0.0,GSE165578,AST01.3,AST01.3_GRIA2.OLFM2,Lin
v3.1.1_CCTCCTCAGCTTTCTT,v3.1.1,2397.519236,1638,0.000000,v3.1.1,cjacchus,CJH01,F,2051,03fWM,...,fWM_He.Control,Hi4000,10xv3,1st,0.0,0.0,GSE165578,AST03,AST03_CPAMD8.HS3ST3A1,Lin
v3.1.1_AACCTGAGTGGTATGG,v3.1.1,5555.186882,2779,0.000000,v3.1.1,cjacchus,CJH01,F,2051,03fWM,...,fWM_He.Control,Hi4000,10xv3,1st,0.0,0.0,GSE165578,AST03,AST03_CPAMD8.HS3ST3A1,Lin
v3.1.1_GAATCACGTAGACGGT,v3.1.1,6184.889256,2770,0.025383,v3.1.1,cjacchus,CJH01,F,2051,03fWM,...,fWM_He.Control,Hi4000,10xv3,1st,0.0,0.0,GSE165578,AST01.3,AST01.3_GRIA2.OLFM2,Lin
v3.1.1_CTCATTAAGGAACGTC,v3.1.1,5421.753017,2666,0.000000,v3.1.1,cjacchus,CJH01,F,2051,03fWM,...,fWM_He.Control,Hi4000,10xv3,1st,0.0,0.0,GSE165578,AST03,AST03_CPAMD8.HS3ST3A1,Lin


In [50]:
assert isinstance(adata.obs, pd.DataFrame)

adata.obs["sample"] = adata.obs["IL01_uniqueID"]
adata.obs["donor"] = adata.obs["IL03_source"]
adata.obs["sex"] = adata.obs["IL04_sex"]
adata.obs["age"] = adata.obs["IL05_ageDays"].astype(float)
adata.obs["brain"] = adata.obs["IL06_tissue.dev"]
adata.obs["matter"] = adata.obs["IL06_tissue.coarse"]
adata.obs["tissue"] = adata.obs["IL06_tissue.fine"]
adata.obs["lesion"] = adata.obs["IL08_condition"]
adata.obs["lesion_age"] = adata.obs["IL08_condition.maxDay"]
adata.obs["condition"] = adata.obs["IL08_condition.status"]
adata.obs["pool"] = adata.obs["IL11_batch"]
adata.obs["lesion_age_min"] = adata.obs["IL12_LMinDays"].astype(float)
adata.obs["lesion_age_max"] = adata.obs["IL13_LMaxDays"].astype(float)
adata.obs["celltype"] = adata.obs["IL15_annotation"]
adata.obs["series"] = adata.obs["series"]

age_mean = adata.obs["age"].mean()
age_sd = adata.obs["age"].std()
adata.obs["age_scale"] = (adata.obs["age"] - age_mean) / (2 * age_sd)

keep = ["sample", "donor", "sex", "brain", "matter", "tissue", "lesion", "lesion_age", "condition", "pool", "lesion_age_min", "lesion_age_max", "celltype", "series"]
adata.obs.drop(columns = [c for c in adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [52]:
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
adata.write_h5ad(os.path.join(h5ad_matrix_location, SAVE_FILE_NAME))
del adata
gc.collect()

5803